# Target Return Portfolio (Efficient Frontier Point)

**Objective key:** `target_return` &nbsp;·&nbsp; **Family:** Classical &nbsp;·&nbsp; **Speed:** Fast

## Algorithm

Finds the minimum-variance portfolio that achieves at least a specified return target.  
This is a single point on the efficient frontier parameterised by the target return.  
Internally, it minimises `wᵀΣw` subject to `wᵀμ ≥ target_return` and weight bounds.

## Papers

- **Foundational:** Markowitz (1952) — *Portfolio Selection* — Journal of Finance  
  https://doi.org/10.1111/j.1540-6261.1952.tb01525.x
- **Modern:** Boyd & Vandenberghe (2004) — *Convex Optimization* §4.4 (parametric QP, free online)  
  https://web.stanford.edu/~boyd/cvxbook/

## Assumptions

- `mu` and `Sigma` are annualised.
- `target_return` must lie between the minimum and maximum feasible returns.  
  Below or above that range the solver will raise an error.
- `weight_min=0.005`, `weight_max=0.30`, `seed=42`.

In [ ]:
import sys
from pathlib import Path

def _find_root() -> Path:
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (p / "api.py").is_file():
            return p
    raise RuntimeError("Cannot find repo root")

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"Repo root: {ROOT}")

In [ ]:
import numpy as np
from services.portfolio_optimizer import run_optimization

rng = np.random.default_rng(42)
n = 10
mu = rng.uniform(0.06, 0.22, n)
cov_raw = rng.standard_normal((n, n))
Sigma = cov_raw @ cov_raw.T / n + np.eye(n) * 0.04
asset_names = [f"ASSET_{i+1:02d}" for i in range(n)]

# Choose a target between min and max achievable return
min_ret = mu.min()
max_ret = mu.max()
target = float(mu.mean())  # midpoint of asset return range
print(f"Asset return range: [{min_ret:.4f}, {max_ret:.4f}]")
print(f"Target return:       {target:.4f}")

In [ ]:
result = run_optimization(
    returns=mu,
    covariance=Sigma,
    objective="target_return",
    target_return=target,
    asset_names=asset_names,
    weight_min=0.005,
    weight_max=0.30,
    seed=42,
)

print(f"\nObjective:        {result.objective}")
print(f"Target return:    {target:.4f}")
print(f"Achieved return:  {result.expected_return:.4f}")
print(f"Volatility:       {result.volatility:.4f}")
print(f"Sharpe ratio:     {result.sharpe_ratio:.4f}")
assert result.expected_return >= target - 1e-4, "Return constraint not satisfied"
print("\nReturn constraint satisfied ✓")
print("\nWeights:")
for name, w in zip(asset_names, result.weights):
    if w > 0.01:
        print(f"  {name}: {w:.4f}")

In [ ]:
# Sweep three target points to trace part of the efficient frontier
targets = [min_ret + (max_ret - min_ret) * t for t in [0.2, 0.5, 0.8]]
print("Target  |  Return  |  Volatility  |  Sharpe")
print("-" * 50)
for tgt in targets:
    try:
        r = run_optimization(
            returns=mu, covariance=Sigma, objective="target_return",
            target_return=tgt, weight_min=0.005, weight_max=0.30, seed=42
        )
        print(f"{tgt:.4f}  |  {r.expected_return:.4f}  |  {r.volatility:.4f}       |  {r.sharpe_ratio:.4f}")
    except Exception as e:
        print(f"{tgt:.4f}  |  ERROR: {e}")